# SRP — V1 on Published Benchmarks (100k–200k rows)

**Goal:** measure the original **V1** architecture ("MLPs with Attention V1") on standard tabular benchmarks
that have **published results**, so it can be placed against GBDTs and deep-learning baselines
from the literature instead of only against our own runs.

### Datasets and source

All three datasets, their **exact train/val/test splits**, and the published results come from one paper:
Gorishniy, Rubachev & Babenko (NeurIPS 2022), *On Embeddings for Numerical Features in Tabular Deep Learning*
(arXiv:2203.05556). Its appendix reports 33 models: CatBoost, XGBoost, MLP, ResNet and Transformer, each with and
without numerical embeddings. Single models use 15 seeds; ensembles average 5 models.

| Dataset | Rows | Features | Task | Metric | Note |
|---|---|---|---|---|---|
| **Higgs Small** | 98,049 | 28 | binary | accuracy | physics simulation, near-balanced |
| **Facebook Comments** | 197,080 | 50 num + 1 categorical (81 values) | regression | RMSE | comment counts, heavy-tailed |
| **Santander** | 200,000 | 200 | binary | accuracy | **90% one class**: always predicting it scores 0.899 |

### Protocol
* **The paper's exact splits** (verified below) and its preprocessing: quantile-transformed numerical features,
  one-hot categorical feature, standardised regression target. RMSE is reported in original units, as in the paper.
* **V1 is run unchanged:** its own model code (vendored in `v1/`), its own training loop and early stopping, and its
  own default hyperparameters (`hparams.py`: 8 MLPs, d_model 128, 2 layers, CKA penalty 0.4, 30 epochs).
* **5 seeds** per dataset (the paper used 15), plus an **ensemble of those 5**, which is comparable to the paper's
  5-model ensembles.
* **Sanity check:** we re-tune and re-run **XGBoost** on the same splits. If our XGBoost lands near the published
  XGBoost, our data loading and metrics match the paper's, and V1's numbers can be read against the published table.

### Read the comparison with this asymmetry in mind
The published models were **tuned per dataset** (extensive hyperparameter search). V1 runs with **fixed defaults**
designed on small datasets. So this measures *V1 as it stands*; a V1 tuned per dataset could do better.

## 0. Setup

In [ ]:
import os, sys, time, json, warnings
warnings.filterwarnings("ignore")
PROJECT_DIR = os.getcwd()
sys.path.insert(0, PROJECT_DIR)
os.environ["PYTHONPATH"] = PROJECT_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")

import numpy as np, pandas as pd, torch, optuna
import matplotlib.pyplot as plt, seaborn as sns
from joblib import Parallel, delayed
from xgboost import XGBClassifier, XGBRegressor

from benchmarks import BENCHMARKS, load_benchmark, published_results, score, is_higher_better
from v1_runner import run_v1, V1_DEFAULTS, default_device

DATASETS   = ["higgs-small", "fb-comments", "santander"]
SEEDS      = (0, 1, 2, 3, 4)
XGB_TRIALS = 40
SUBSAMPLE  = None        # e.g. 3000 for a quick smoke test; None = full published splits
V1_HP      = {}          # overrides of V1_DEFAULTS, e.g. {"epochs": 2} for a smoke test

DEVICE = default_device()
# Several V1 runs share one GPU well (each needs well under 1 GB); on CPU use small jobs side by side.
THREADS_PER_JOB = 2
N_JOBS = 3 if DEVICE == "cuda" else max(1, min(8, (os.cpu_count() or 2) // THREADS_PER_JOB))

RUN_DIR = os.path.join(PROJECT_DIR, "runs"); os.makedirs(RUN_DIR, exist_ok=True)
LOG_PATH = os.path.join(RUN_DIR, "05_progress.log")
if os.path.exists(LOG_PATH): os.remove(LOG_PATH)
def log(msg):
    print(msg, flush=True)
    with open(LOG_PATH, "a") as f: f.write(time.strftime("%H:%M:%S ") + msg + "\n")

def show(df, fmt=None, style=None):
    try:
        s = df.style.format(fmt or {}, na_rep="—"); display(style(s) if style else s)
    except (AttributeError, ImportError):
        display(df.round(4))

sns.set_theme(style="whitegrid", context="notebook")
optuna.logging.set_verbosity(optuna.logging.WARNING)
gpu = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "none"
log(f"device: {DEVICE} ({gpu}) | torch {torch.__version__} | {N_JOBS} parallel V1 jobs")
print("V1 hyperparameters:", {**V1_DEFAULTS, **V1_HP})

## 1. Load the datasets — and verify they are the paper's exact splits

In [ ]:
TASKS = {}
rows = []
for name in DATASETS:
    t = load_benchmark(name)
    ours = (len(t.y_tr), len(t.y_va), len(t.y_te))
    paper = tuple(t.info["published_split_sizes"])
    assert ours == paper, f"{name}: split sizes {ours} differ from the paper's {paper}"
    if SUBSAMPLE:
        n = SUBSAMPLE
        t = t.with_(X_tr=t.X_tr[:n], y_tr=t.y_tr[:n], X_va=t.X_va[:n // 3], y_va=t.y_va[:n // 3],
                    X_te=t.X_te[:n], y_te=t.y_te[:n])
    TASKS[name] = t
    rows.append({"dataset": BENCHMARKS[name]["label"], "task": t.task, "features": t.n_features,
                 "train / val / test (paper)": " / ".join(f"{v:,}" for v in paper),
                 "matches paper": "yes", "majority-class acc": t.majority_acc})
show(pd.DataFrame(rows).set_index("dataset"), {"majority-class acc": "{:.3f}"})
if SUBSAMPLE:
    print(f"SMOKE TEST: training on {SUBSAMPLE:,} rows per dataset — numbers below are NOT comparable to the paper.")

## 2. The published results

From the paper's appendix (Table 18: single models, mean ± std over 15 seeds). A focused subset is shown:
GBDTs, the plain MLP, ResNet and Transformer, and the best embedding variants. All 33 models are used for V1's rank later.

In [ ]:
PUB = published_results()
FOCUS = ["CatBoost", "XGBoost", "MLP", "MLP-PLR", "ResNet", "ResNet-PLR", "Transformer-L", "Transformer-PLR"]
GBDT = {"CatBoost", "XGBoost"}

def pub_table(kind):
    t = pd.DataFrame({BENCHMARKS[d]["label"]: {m: f"{PUB[kind][m][d]['mean']:.3f} ± {PUB[kind][m][d]['std']:.3f}"
                                               for m in FOCUS if m in PUB[kind]} for d in DATASETS})
    return t
print("Published SINGLE models (higher accuracy / lower RMSE is better):")
display(pub_table("single"))
print("Source:", PUB["source"])

## 3. Sanity check — reproduce XGBoost on the same splits

XGBoost is tuned with Optuna on the **validation** split (40 trials, early stopping on validation, as in the
paper's protocol), then re-trained on 5 seeds. If it lands close to the published XGBoost, the data and
metric pipeline matches the paper's.

In [ ]:
def xgb_model(params, task, seed):
    common = dict(tree_method="hist", device=DEVICE, random_state=seed, n_estimators=2000,
                  early_stopping_rounds=50, **params)
    return XGBRegressor(**common) if task == "regression" else XGBClassifier(**common)

def xgb_raw(m, task, X):
    if task == "regression":
        return m.predict(X)
    p = np.clip(m.predict_proba(X)[:, 1], 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))                          # logit, so score() thresholds at 0

def xgb_suggest(trial):
    return {"max_depth": trial.suggest_int("max_depth", 3, 12),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
            "min_child_weight": trial.suggest_float("min_child_weight", 1e-2, 1e2, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
            "gamma": trial.suggest_float("gamma", 1e-8, 1.0, log=True)}

XGB = {}
for name in DATASETS:
    t, t0 = TASKS[name], time.time()
    hib = is_higher_better(name)
    def objective(trial):
        m = xgb_model(xgb_suggest(trial), t.task, 0).fit(t.X_tr, t.y_tr, eval_set=[(t.X_va, t.y_va)], verbose=False)
        return score(t, t.y_va, xgb_raw(m, t.task, t.X_va))
    st = optuna.create_study(direction="maximize" if hib else "minimize",
                             sampler=optuna.samplers.TPESampler(seed=0, multivariate=True))
    st.optimize(objective, n_trials=XGB_TRIALS)
    scores = []
    for s in SEEDS:
        m = xgb_model(st.best_params, t.task, s).fit(t.X_tr, t.y_tr, eval_set=[(t.X_va, t.y_va)], verbose=False)
        scores.append(score(t, t.y_te, xgb_raw(m, t.task, t.X_te)))
    XGB[name] = np.array(scores)
    pub = PUB["single"]["XGBoost"][name]
    log(f"[xgb] {BENCHMARKS[name]['label']:18s} ours {XGB[name].mean():.4f} ± {XGB[name].std(ddof=1):.4f} | "
        f"published {pub['mean']:.3f} ± {pub['std']:.3f}  ({(time.time()-t0)/60:.1f} min)")

## 4. Train V1 — 5 seeds per dataset

The long cell. Each run is V1's own `train_joint` (up to 30 epochs, early stopping patience 10). Predictions on
the full test set are kept so the 5 seeds can also be averaged into an ensemble.

In [ ]:
jobs = [(name, s) for name in ["fb-comments", "santander", "higgs-small"] if name in DATASETS for s in SEEDS]
log(f"[v1] {len(jobs)} runs, {N_JOBS} at a time on {DEVICE} ...")
t0, V1 = time.time(), []
for (name, s), r in zip(jobs, Parallel(n_jobs=N_JOBS, return_as="generator")(
        delayed(run_v1)(TASKS[name], s, DEVICE, V1_HP, False, THREADS_PER_JOB) for name, s in jobs)):
    t = TASKS[name]
    r.update(dataset=name, test_score=score(t, t.y_te, r["test_pred"]), val_score=score(t, t.y_va, r["val_pred"]))
    V1.append(r)
    log(f"[v1] {BENCHMARKS[name]['label']:18s} seed {s}  test {BENCHMARKS[name]['metric']} {r['test_score']:.4f}  "
        f"({r['epochs_run']} epochs, {r['seconds']/60:.1f} min)  elapsed {(time.time()-t0)/60:.1f} min")
log(f"[v1] all runs done in {(time.time()-t0)/60:.1f} min")

def ensemble_score(name):
    t = TASKS[name]
    preds = [r["test_pred"] for r in V1 if r["dataset"] == name]
    if t.task == "regression":
        return score(t, t.y_te, np.mean(preds, axis=0))
    p = np.mean([1 / (1 + np.exp(-np.asarray(x).reshape(-1))) for x in preds], axis=0)   # average probabilities
    return score(t, t.y_te, np.log(np.clip(p, 1e-7, 1 - 1e-7) / np.clip(1 - p, 1e-7, 1)))
V1_ENS = {name: ensemble_score(name) for name in DATASETS}

## 5. Results

In [ ]:
v1 = pd.DataFrame([{k: r[k] for k in ("dataset", "seed", "test_score", "val_score", "epochs_run", "seconds", "n_params")}
                   for r in V1])

def rank_among(name, value, kind):
    # 1 = best. Rank of `value` among all published models of that kind on this dataset.
    vals = [PUB[kind][m][name]["mean"] for m in PUB[kind]]
    better = sum((v > value) if is_higher_better(name) else (v < value) for v in vals)
    return better + 1, len(vals) + 1

rows = []
for name in DATASETS:
    lab, hib = BENCHMARKS[name]["label"], is_higher_better(name)
    x = v1[v1.dataset == name]["test_score"]
    best_gbdt = (max if hib else min)(PUB["single"][m][name]["mean"] for m in GBDT)
    best_nn = (max if hib else min)(PUB["single"][m][name]["mean"] for m in PUB["single"] if m not in GBDT)
    r_s, n_s = rank_among(name, x.mean(), "single")
    r_e, n_e = rank_among(name, V1_ENS[name], "ensemble")
    sign = 1 if hib else -1
    rows.append({"dataset": lab, "metric": BENCHMARKS[name]["metric"],
                 "V1 single (5 seeds)": f"{x.mean():.4f} ± {x.std(ddof=1):.4f}",
                 "V1 ensemble of 5": f"{V1_ENS[name]:.4f}",
                 "published MLP": PUB["single"]["MLP"][name]["mean"],
                 "best published GBDT": best_gbdt, "best published DL": best_nn,
                 "V1 − MLP": sign * (x.mean() - PUB["single"]["MLP"][name]["mean"]),
                 "V1 − best GBDT": sign * (x.mean() - best_gbdt),
                 "V1 rank (single)": f"{r_s} / {n_s}", "V1 ensemble rank": f"{r_e} / {n_e}",
                 "our XGBoost repro": f"{XGB[name].mean():.4f} ± {XGB[name].std(ddof=1):.4f}",
                 "published XGBoost": f"{PUB['single']['XGBoost'][name]['mean']:.3f} ± {PUB['single']['XGBoost'][name]['std']:.3f}"})
SUMMARY = pd.DataFrame(rows).set_index("dataset")
print("Differences are signed so that POSITIVE = V1 is better (for RMSE the sign is flipped).")
show(SUMMARY, {"published MLP": "{:.3f}", "best published GBDT": "{:.3f}", "best published DL": "{:.3f}",
               "V1 − MLP": "{:+.4f}", "V1 − best GBDT": "{:+.4f}"})

print("\nPer-seed V1 runs")
show(v1.drop(columns="n_params").assign(minutes=lambda d: d.seconds / 60).drop(columns="seconds"),
     {"test_score": "{:.4f}", "val_score": "{:.4f}", "minutes": "{:.1f}"})

### 5.1 Where V1 sits among the published models

In [ ]:
fig, axes = plt.subplots(1, len(DATASETS), figsize=(6 * len(DATASETS), 6.2))
for ax, name in zip(np.atleast_1d(axes), DATASETS):
    hib = is_higher_better(name)
    entries = [(m, PUB["single"][m][name]["mean"], PUB["single"][m][name]["std"],
                "#ff8a3d" if m in GBDT else "#9aa0a6") for m in FOCUS]
    x = v1[v1.dataset == name]["test_score"]
    entries.append(("V1 (ours, 5 seeds)", x.mean(), x.std(ddof=1), "#7b5cff"))
    entries.append(("V1 ensemble of 5", V1_ENS[name], 0.0, "#4a2fd6"))
    entries.append(("XGBoost (our repro)", XGB[name].mean(), XGB[name].std(ddof=1), "#ffc08a"))
    entries.sort(key=lambda e: e[1], reverse=not hib)                     # best at the top
    for i, (lab, m, sd, c) in enumerate(entries):
        ax.errorbar(m, i, xerr=sd, fmt="o", color=c, ms=9 if lab.startswith("V1") else 7,
                    capsize=4, mec="black" if lab.startswith("V1") else c)
        ax.text(m, i + 0.32, f"{m:.3f}", ha="center", fontsize=8, color=c if c != "#9aa0a6" else "#555")
    ax.set_yticks(range(len(entries))); ax.set_yticklabels([e[0] for e in entries], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel(f"test {BENCHMARKS[name]['metric']}  ({'higher' if hib else 'lower'} is better)")
    ax.set_title(BENCHMARKS[name]["label"], fontweight="bold")
    if name == "santander":
        ax.text(0.01, 0.01, "always predicting the majority class = 0.899", transform=ax.transAxes, fontsize=8, color="#777")
plt.suptitle("V1 against published single models (orange = GBDT, grey = deep learning, purple = V1)",
             fontweight="bold", y=1.02)
plt.tight_layout(); plt.show()

## 6. Findings

*(Computed from the runs above.)*

In [ ]:
sep = "=" * 92
print(sep); print("FINDINGS — V1 on published benchmarks"); print(sep)
if SUBSAMPLE:
    print("SMOKE TEST MODE — trained on a subsample; these numbers are not comparable to the paper.\n")

print("\n1. Does our pipeline reproduce the paper? (XGBoost, ours vs published)")
for name in DATASETS:
    pub = PUB["single"]["XGBoost"][name]
    d = XGB[name].mean() - pub["mean"]
    tol = 2 * np.sqrt(pub["std"] ** 2 + XGB[name].std(ddof=1) ** 2) + (0.002 if is_higher_better(name) else 0.05)
    print(f"   {BENCHMARKS[name]['label']:18s} ours {XGB[name].mean():.4f} vs published {pub['mean']:.3f}  "
          f"(diff {d:+.4f}) -> " + ("reproduced" if abs(d) <= tol else "NOT reproduced — compare with care"))

print("\n2. V1 per dataset (single model, mean over seeds; positive = V1 better)")
for name in DATASETS:
    r = SUMMARY.loc[BENCHMARKS[name]["label"]]
    print(f"   {BENCHMARKS[name]['label']:18s} V1 {r['V1 single (5 seeds)']}  | vs MLP {r['V1 − MLP']:+.4f} | "
          f"vs best GBDT {r['V1 − best GBDT']:+.4f} | rank {r['V1 rank (single)']} among published single models")

print("\n3. V1 ensemble of 5 vs published 5-model ensembles")
for name in DATASETS:
    r = SUMMARY.loc[BENCHMARKS[name]["label"]]
    print(f"   {BENCHMARKS[name]['label']:18s} V1 ensemble {r['V1 ensemble of 5']}  rank {r['V1 ensemble rank']}")

beats_mlp = sum(SUMMARY["V1 − MLP"] > 0)
beats_gbdt = sum(SUMMARY["V1 − best GBDT"] > 0)
print(f"\n4. Overall: V1 beats the published plain MLP on {beats_mlp}/{len(DATASETS)} datasets and the best "
      f"published GBDT on {beats_gbdt}/{len(DATASETS)}.")
print(sep)

## 7. Caveats and next steps

* **Tuning asymmetry.** Published models were tuned per dataset; V1 used fixed defaults. A fair head-to-head
  would tune V1 too (the `tuning.py` Optuna machinery can be pointed at V1).
* **Seeds.** 5 seeds here vs 15 in the paper, so V1's standard deviations are rougher estimates.
* **Adding our v4 models** (tuned mean-pool and causal attention from notebook 03) to this same comparison is a
  small change: they share the data loader and the metric.